# Subclustering Mesenchymal cells

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
library(stringr)
set.seed(1234)

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4', '#f5674a', '#C79FEF', '#DAA520', '#A0522D')

# Data loading

In [ ]:
annotation <- import('genes2.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style

In [ ]:
annotation$tx_id = annotation$transcript_id

In [ ]:
#load fragments data
P11_sample1_frags <- 'P11_sample1_atac_fragments.tsv.gz'

P11_sample2_frags <- 'P11_sample2_atac_fragments.tsv.gz'

P11_sample3_frags <- 'P11_sample3_atac_fragments.tsv.gz'

P60_sample1_frags <- 'P60_sample1_atac_fragments.tsv.gz'

P60_sample2_frags <- 'P60_sample2_atac_fragments.tsv.gz'

P60_TTR_sample3_frags <- 'P60_TTR_sample3_atac_fragments.tsv.gz'

In [ ]:
sample_names <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3')

In [ ]:
frags_path <- c(P11_sample1_frags, P11_sample2_frags, P11_sample3_frags, P60_sample1_frags, P60_sample2_frags, 
           P60_TTR_sample3_frags)

In [ ]:
raw <- readRDS('Merged_multiome.rds')

In [ ]:
filtered <- readRDS('Integrated_multiome_clusters_joint_linked_cell_type.rds')

In [ ]:
raw

In [ ]:
filtered

# Subset data into only Mesenchymal cells

In [ ]:
mesenchymal_filt <- subset(filtered, subset = predicted.id == 'Mesenchymal cells')

In [ ]:
mesenchymal_filt

In [ ]:
mesenchymal <- subset(raw, cells = rownames(mesenchymal_filt@meta.data))

In [ ]:
mesenchymal

In [ ]:
frags <- Fragments(mesenchymal)

In [ ]:
Fragments(mesenchymal) <- NULL

In [ ]:
for (i in 1:length(frags)){
    frags[[i]] <- UpdatePath(frags[[i]], new.path = frags_path[[i]])
}

In [ ]:
Fragments(mesenchymal) <- frags

In [ ]:
saveRDS(mesenchymal, 'Mesenchymal_multiome.rds')

In [ ]:
mesenchymal <- readRDS('Mesenchymal_multiome.rds')

In [ ]:
mesenchymal@assays

In [ ]:
unique(mesenchymal$Sample)

In [ ]:
unique(mesenchymal$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
mesenchymal$nucleosome_group <- ifelse(mesenchymal$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = mesenchymal, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="Mesenchymal_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Mesenchymal_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Filtering

In [ ]:
table(mesenchymal$orig.ident)

In [ ]:
DefaultAssay(mesenchymal) <- "RNA"

Since we already have the peaks assay we can delete the ATAC assay, which allows to save compuattional resources 

In [ ]:
samples.list <- SplitObject(mesenchymal, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
mesenchymal <- mesenchymal %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(mesenchymal, ndims = 50)

pca

ggsave(file="Mesenchymal_bff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
mesenchymal <- RunUMAP(mesenchymal, dims = 1:12, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(mesenchymal, reduction = 'umap', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

dpl

ggsave(file="Mesenchymal_bff_RNA_UMAP.svg", plot=dpl, width=8, height=4)

**NOTE**:The functions *get_density* and *DensityScatter* are from the packages Signac V 1.10, since we are using the version 1.4 we copied only the necessary function to plot the wanted plots.

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)
    # logfny <- null_fn
    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
nCount_RNA_max <- c(5200, 6500, 7000, 7000, 10000, 1200)

nCount_RNA_min <- c(250, 250, 250, 250, 250, 250)

nFeature_RNA_max <- c(2900, 3000, 3500, 3000, 6000, 900)

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(14700, 18000, 15000, 13000, 25000, 5800)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200)

nFeature_ATAC_max <- c(7000, 7800, 6500, 6000, 10000, 3200)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.6, 1.2)

pct_hb_max <- c(0.075, 0.075, 0.06, 0.05, 0.06, 0.05)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(mesenchymal, subset = Sample == sample)

    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
length(cells_to_keep)

In [ ]:
mesenchymal

In [ ]:
mesenchymal_filt <- subset(mesenchymal, cells = cells_to_keep)

In [ ]:
length(colnames(mesenchymal_filt))

In [ ]:
table(mesenchymal_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Mesenchymal_all_vln_after_filter.svg", plot=vln, width=15, height=8)


In [ ]:
saveRDS(mesenchymal_filt, 'Mesenchymal_filt_multiome.rds')

# Call peaks

In [ ]:
mesenchymal_filt <- readRDS('Mesenchymal_filt_multiome.rds')

In [ ]:
mesenchymal_filt

In [ ]:
DefaultAssay(mesenchymal_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=mesenchymal_filt,
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(mesenchymal_filt), features = peaks, cells = colnames(mesenchymal_filt))

In [ ]:
mesenchymal_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(mesenchymal_filt), annotation = annotation)

In [ ]:
mesenchymal_filt@assays

In [ ]:
DefaultAssay(mesenchymal_filt) <- 'peaks'

In [ ]:
mesenchymal_filt[['ATAC']] <- NULL

In [ ]:
saveRDS(mesenchymal_filt, 'Mesenchymalmesenchymal_filt_multiome_peaks.rds')

# FRiP

In [ ]:
mesenchymal_filt <- FRiP(mesenchymal_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Mesenchymalmesenchymal_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(mesenchymal_filt, 'Mesenchymalmesenchymal_filt_multiome_peaks.rds')

# Downstream analysis

In [ ]:
mesenchymal_filt <- readRDS('Mesenchymalmesenchymal_filt_multiome_peaks.rds')

In [ ]:
DefaultAssay(mesenchymal_filt) <- "RNA"

## RNA

In [ ]:
mesenchymal_filt@assays

In [ ]:
table(mesenchymal_filt$orig.ident)

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
mesenchymal_filt <- mesenchymal_filt %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData(vars.to.regress = vars_to_regress) %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(mesenchymal_filt, ndims = 50)

pca

ggsave(file="Mesenchymal_aff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
mesenchymal_filt <- RunUMAP(mesenchymal_filt, dims = 1:14, reduction = 'pca', n.neighbors = 15, reduction.name="umap_RNA")

In [ ]:
unique(mesenchymal_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(mesenchymal_filt, reduction = 'umap_RNA', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="Mesenchymal_aff_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(mesenchymal_filt, reduction = 'umap_RNA', group.by = 'Age', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Mesenchymal_aff_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(mesenchymal_filt, reduction = 'umap_RNA', group.by = 'Batch', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Mesenchymal_aff_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(mesenchymal_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated

In [ ]:
integrated <- RunUMAP(integrated, reduction = 'harmonyRNA', dims = 1:14, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)


In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, cols=my_c, ncol=3, 
               pt.size=2.5, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

### Clustering

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:14) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Mesenchymal_integrated_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.1')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

In [ ]:
saveRDS(integrated, 'Mesenchymal_integrated_multiome_clusters.rds')

# ATAC

In [ ]:
integrated <- readRDS('Mesenchymal_integrated_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- "peaks"

In [ ]:
integrated <- FindTopFeatures(integrated, assay = "peaks", min.cutoff = 10)

In [ ]:
integrated <- RunTFIDF(integrated, assay="peaks")

In [ ]:
integrated <- RunSVD(integrated, assay="peaks")

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
dpc <- DepthCor(integrated, n=50)

dpc

ggsave(file="Merged_depthCorr.svg", plot=dpc, width=6, height=4)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
ebl <- ElbowPlot(integrated, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated <- FindNeighbors(integrated, reduction = 'lsi', dims = 2:9, graph.name="snn_Peaks")

In [ ]:
integrated <- RunUMAP(integrated, dims = 2:9, reduction = 'lsi', reduction.name="umap_Peaks")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Sample", reduction="umap_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="Merged_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="umap_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=2.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Merged_Peaks_each_sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Batch", reduction="umap_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_Peaks_Batch_UMAP.svg", plot=dpl, width=7, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Age", reduction="umap_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_Peaks_Age_UMAP.svg", plot=dpl, width=7, height=4)


### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(integrated, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'Peaks', reduction.save = 'harmonyPeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated <- RunUMAP(integrated, dims = 2:9, reduction = 'harmonyPeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by = 'Sample', pt.size = 2.5, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="Mesenchymal_integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=5)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=2.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_Peaks_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Batch", reduction="harmony_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_Peaks_Batch_UMAP.svg", plot=dpl, width=7, height=5)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Age", reduction="harmony_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_Peaks_Age_UMAP.svg", plot=dpl, width=7, height=5)

### Clustering

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
integrated

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyPeaks', dims = 2:9) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Mesenchymal_integrated_Peaks_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated, 'Mesenchymal_integrated_multiome_clusters.rds')

# Cell type annotation

In [ ]:
integrated <- readRDS('Mesenchymal_integrated_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.1')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Ttr', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = marker_genes, pt.size=0.5, cols = my_c, ncol = 3)

vln

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = marker_genes, 
             min.cutoff = 0, max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


In [ ]:
mesenchymal_markers = c('Vcam1', 'Moxd1', 'Col26a1', 'Tcf21',
                  'Top2a', 'Pclaf', 'Birc5', 'Mki67',
                  'Timp1', 'Clec3b', 'Slc6a13', 'Sidt1',
                  'Abcc9', 'Kcnj8', 'Trpc3', 'Serpini1',
                  'Acta2', 'Tagln', 'Myh11', 'Dgkb')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = mesenchymal_markers, pt.size=0.8, cols = my_c, ncol = 4)

vln

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = mesenchymal_markers, min.cutoff = 0, max.cutoff = 4, label.size = 2, 
                   ncol = 4, pt.size = 2.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "RNA_snn_res.1", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Mesenchymal_integrated_clusters_RNA.svg", plot=dpt, width=7, height=5)

# Remove clusters of doublets

In [ ]:
integrated <- subset(integrated, , idents = c(9, 11), invert = TRUE)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "RNA_snn_res.1", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Mesenchymal_integrated_dblRemoved_clusters_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated

In [ ]:
mesenchymal_noDBL <- subset(mesenchymal, cells=rownames(integrated@meta.data))

In [ ]:
mesenchymal_noDBL

In [ ]:
saveRDS(mesenchymal_noDBL, 'Mesen_noDBL_multiome.rds')

# Redo the analysis without doublets removed manually

In [ ]:
mesenchymal_noDBL <- readRDS('Mesen_noDBL_multiome.rds')

In [ ]:
mesenchymal_noDBL

In [ ]:
unique(mesenchymal_noDBL$Sample)

In [ ]:
unique(mesenchymal_noDBL$Age)

In [ ]:
DefaultAssay(mesenchymal_noDBL) <- 'ATAC'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
mesenchymal_noDBL$nucleosome_group <- ifelse(mesenchymal_noDBL$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = mesenchymal_noDBL, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="mesenchymal_noDBL_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal_noDBL, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="mesenchymal_noDBL_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
DefaultAssay(mesenchymal_noDBL) <- 'RNA'

In [ ]:
DefaultAssay(mesenchymal_noDBL) <- 'RNA'

options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal_noDBL, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_noDBL_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Filtering

In [ ]:
table(mesenchymal_noDBL$orig.ident)

In [ ]:
DefaultAssay(mesenchymal_noDBL) <- "RNA"

In [ ]:
samples.list <- SplitObject(mesenchymal_noDBL, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
mesenchymal_noDBL <- mesenchymal_noDBL %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(mesenchymal_noDBL, ndims = 50)

pca

ggsave(file="mesenchymal_noDBL_pca.svg", plot=pca, width=6, height=4)

In [ ]:
mesenchymal_noDBL <- RunUMAP(mesenchymal_noDBL, dims = 1:12, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(mesenchymal_noDBL, reduction = 'umap', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

dpl

ggsave(file="mesenchymal_noDBL_RNA_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)
    # logfny <- null_fn
    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal_noDBL, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(mesenchymal_noDBL, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
nCount_RNA_max <- c(5000, 6000, 6000, 8000, 7000, 1200)

nCount_RNA_min <- c(250, 250, 250, 250, 250, 250)

nFeature_RNA_max <- c(3000, 2900, 3000, 3500, 4000, 1000)

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(12500, 12500, 170000, 15000, 21000, 6000)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200)

nFeature_ATAC_max <- c(5800, 6000, 9000, 8000, 9000, 4000)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.7, 1.2)

pct_hb_max <- c(0.075, 0.075, 0.06, 0.05, 0.05, 0.05)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(mesenchymal_noDBL, subset = Sample == sample)

    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_noDBL_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
mesenchymal_noDBL_filt <- subset(mesenchymal_noDBL, cells = cells_to_keep)

In [ ]:
table(mesenchymal_noDBL_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal_noDBL_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="mesenchymal_noDBL_all_vln_after_filter.svg", plot=vln, width=15, height=8)

In [ ]:
saveRDS(mesenchymal_noDBL_filt, 'Mesen_noDBL_filt_multiome.rds')

# Call peaks

In [ ]:
mesenchymal_noDBL_filt <- readRDS('Mesen_noDBL_filt_multiome.rds')

In [ ]:
mesenchymal_noDBL_filt

In [ ]:
DefaultAssay(mesenchymal_noDBL_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=mesenchymal_noDBL_filt, 
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(mesenchymal_noDBL_filt), features = peaks, cells = colnames(mesenchymal_noDBL_filt))

In [ ]:
mesenchymal_noDBL_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(mesenchymal_noDBL_filt), 
                                             annotation = annotation)

In [ ]:
DefaultAssay(mesenchymal_noDBL_filt) <- 'peaks'

In [ ]:
mesenchymal_noDBL_filt[['ATAC']] <- NULL

# FRiP

In [ ]:
mesenchymal_noDBL_filt <- FRiP(mesenchymal_noDBL_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:


options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(mesenchymal_noDBL_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Mesenchymal_mesenchymal_noDBL_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(mesenchymal_noDBL_filt, 'Mesenchymal_noDBL_mesenchymal_filt_multiome_peaks.rds')

# Downstream analysis

In [ ]:
mesenchymal_noDBL_filt <- readRDS('Mesenchymal_noDBL_mesenchymal_filt_multiome_peaks.rds')

## RNA

In [ ]:
DefaultAssay(mesenchymal_noDBL_filt) <- "RNA"

In [ ]:
mesenchymal_noDBL_filt@assays

In [ ]:
table(mesenchymal_noDBL_filt$orig.ident)

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
mesenchymal_noDBL_filt <- mesenchymal_noDBL_filt %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData(vars.to.regress = vars_to_regress) %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(mesenchymal_noDBL_filt, ndims = 50)

pca

ggsave(file="mesenchymal_noDBL_aff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
mesenchymal_noDBL_filt <- RunUMAP(mesenchymal_noDBL_filt, dims = 1:11, reduction = 'pca', n.neighbors = 15, reduction.name="umap_RNA")

In [ ]:
unique(mesenchymal_noDBL_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(mesenchymal_noDBL_filt, reduction = 'umap_RNA', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="mesenchymal_noDBL_aff_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(mesenchymal_noDBL_filt, reduction = 'umap_RNA', group.by = 'Age', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="mesenchymal_noDBL_aff_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(mesenchymal_noDBL_filt, reduction = 'umap_RNA', group.by = 'Batch', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="mesenchymal_noDBL_aff_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated_noDBL <- RunHarmony(mesenchymal_noDBL_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, reduction = 'harmonyRNA', dims = 1:11, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 2.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_noDBL_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_noDBL_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 2.5, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_noDBL_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=2.5, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_noDBL_UMAP.svg", plot=dpl, width=24, height=12)

## Clustering

In [ ]:
integrated_noDBL

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:


integrated_noDBL <- integrated_noDBL %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:11) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.1, 1.2, 1.3,
                                                           1.4, 2.5, 1.6))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.2.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=2.5, group.by = 'RNA_snn_res.1.6', cols = my_c, raster = TRUE))
grid

ggsave(file="Mesenchymal_integrated_noDBL_clusters_all_noDBL.svg", plot=grid, width=15, height=30)

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'RNA_snn_res.1.2')

In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_noDBL_multiome_peaks_clusters.rds')

## ATAC

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_peaks_clusters.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- "peaks"

In [ ]:
integrated_noDBL <- FindTopFeatures(integrated_noDBL, assay = "peaks", min.cutoff = 10)

In [ ]:
integrated_noDBL <- RunTFIDF(integrated_noDBL, assay="peaks")

In [ ]:
integrated_noDBL <- RunSVD(integrated_noDBL, assay="peaks")

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
dpc <- DepthCor(integrated_noDBL, n=50)

dpc

ggsave(file="Merged_depthCorr_noDBL.svg", plot=dpc, width=6, height=4)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
ebl <- ElbowPlot(integrated_noDBL, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl_noDBL.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated_noDBL <- FindNeighbors(integrated_noDBL, reduction = 'lsi', dims = 2:14, graph.name="snn_peaks")

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, dims = 2:14, reduction = 'lsi', reduction.name="umap_peaks")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Sample", reduction="umap_peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="Merged_peaks_Sample_UMAP_noDBL.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="umap_peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=2.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Merged_peaks_each_sample_UMAP_noDBL.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Batch", reduction="umap_peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Batch_UMAP_noDBL.svg", plot=dpl, width=7, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Age", reduction="umap_peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Age_UMAP_noDBL.svg", plot=dpl, width=7, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated_noDBL <- RunHarmony(integrated_noDBL, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonypeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated_noDBL <- RunUMAP(integrated_noDBL, dims = 2:14, reduction = 'harmonypeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by = 'Sample', pt.size = 2.5, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="Mesenchymal_integrated_noDBL_peaks_noDBL_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, 
               ncol=3, pt.size=2.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_noDBL_peaks_noDBL_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated_noDBL, group.by="Batch", reduction="harmony_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_noDBL_Peaks_noDBL_Batch_UMAP.svg", plot=dpl, width=7, height=5)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated_noDBL, group.by="Age", reduction="harmony_Peaks", pt.size = 2.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Mesenchymal_integrated_noDBL_Peaks_noDBL_Age_UMAP.svg", plot=dpl, width=7, height=5)

### Clustering

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- integrated_noDBL %>% FindNeighbors(reduction = 'harmonypeaks', dims = 2:14) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=2.5, group.by = 'peaks_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Mesenchymal_integrated_noDBL_Peaks_noDBL_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_noDBL_multiome_clusters.rds')

# Cell type annotation

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'RNA_snn_res.1.2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = marker_genes, pt.size=0.5, cols = my_c, ncol = 3)

vln

ggsave(file="Mesenchymal_integrated_noDBL_vln_Cell_type.svg", plot=vln, width=10, height=25)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = marker_genes, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
fb_markers = c('Pdgfra', 'Col1a1', 'Lum', 'Acta2',
               'Slit2', 'Igfbp5', 'Ntn1', 'Plxna4', 'Egfr', 'Phactr1', 'Cd9',
               'Slc6a1', 'Atp1a2', 'Slc6a13', 'Slc6a12', 'Igf2bp2',
               'Igfbp7', 'Igfbp4',
               'Igf2bp2', 'Igfbp6', 'Slc6a12',
               'Rgs5', 'Ndufa412', 'Kcnj8', 'Serpini1',
               'Actc1', 'Mustn1', 'Tagln', 'Myh11', 'Map3k7cl', 'Eln', 'Itga8', 'Nov')

In [ ]:


mesenchymal_noDBL_markers = c('Vcam1', 'Moxd1', 'Col26a1', 'Tcf21',
                  'Top2a', 'Pclaf', 'Birc5', 'Mki67',
                  'Timp1', 'Clec3b', 'Slc6a13', 'Sidt1',
                  'Abcc9', 'Kcnj8', 'Trpc3', 'Serpini1',
                  'Acta2', 'Tagln', 'Myh11', 'Dgkb')



In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = mesenchymal_noDBL_markers, pt.size=0.8, cols = my_c, ncol = 4)

vln

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = mesenchymal_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
options(repr.plot.width = 20, repr.plot.height = 30, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = fb_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
pietila_markers = c('Slc47a1', 'Col1a1', 'Adcyap1', 'Pcdh17', 'Calca',
                    'Slc47a1', 'Cdh5', 'Cdh11', 'Slc26a7', 'Atp1b1', 'Slc23a2',
                    'Dpp4', 'Slc7a14', 'Cldn11', 'Cav2', 'Cav1',
                    'Ppp1r1a', 'Prox1', 'Slc6a1', 'Slc22a8', 'Arhgap9',
                    'Slc7a11', 'Sidt1', 'Nespas', 'Coch',
                    'Lama1', 'Col15a1', 'Ngfr', 'Vtn', 'Flrt2', 'Slc6a13', 'Fbln2', 'Slc30a10', 'Mgp', 'Serpine2', 'Clec3b', 'Spp1', 'Col12a1', 'Col4a2',
                    'Tcf21', 'Mylk', 'Tbc1d2b', 'Tmem176a', 'Dach1', 'Col26a1', 'Emcn', 'Lsamp', 'Lamc3')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 40, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = pietila_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
barrier_fb <- c('Igfbp6', 'S100b', 'Cldn11', 'Gjb2', 'Gjb6', 'Cdh1', 'Foxp2', 'Fut8', 'Chrm3', 'Cdh5')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = barrier_fb, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
top10_predicted_noDBL <- top10_predicted_noDBL[!duplicated(top10_predicted_noDBL$gene), ]

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted_noDBL$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "RNA_snn_res.0.2", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

In [ ]:
integrated_noDBL@meta.data$RNA_snn_res.0.2 <- as.factor(integrated_noDBL@meta.data$RNA_snn_res.0.2)

In [ ]:
integrated_noDBL@meta.data$Cell_type <- NA

In [ ]:
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.2 == "0")] <- "Fibroblasts" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.2 == "1")] <- "Pericytes" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.2 == "2")] <- "Fibroblasts" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.2 == "3")] <- "VSMC"
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.2 == "4")] <- "Proliferating mesenchymal cells" 

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'Cell_type')

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Mesenchymal_integrated_noDBL_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
write.csv(integrated_noDBL_predicted.markers, 'Markers_RNA_Mesenchymal.csv', row.names = TRUE)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
top10_predicted_noDBL <- top10_predicted_noDBL[!duplicated(top10_predicted_noDBL$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted_noDBL$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="Mesenchymal_integrated_noDBL_dotplot_Cell_type.svg", plot=dtp, width=10, height=6)

In [ ]:
mesenchymal_noDBL_markers <- c('Vcam1', 'Rbfox1', 'Col26a1', 'Cemip',
                         'Acta2', 'Tagln', 'Myh11', 'Dgkb',
                         'Abcc9', 'Kcnj8', 'Zfp804b', 'Serpini1',
                         'Top2a', 'Pclaf', 'Diaph3', 'Mki67')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 20, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = mesenchymal_noDBL_markers, pt.size=0, cols = my_c, ncol = 4)

vln

ggsave(file="Mesenchymal_integrated_noDBL_vln_Cell_type.svg", plot=vln, width=15, height=20)

In [ ]:
integrated_noDBL

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_multiome_celltype.rds')

In [ ]:
markers <- c('Vcam1', 'Rbfox1', 'Col26a1', 'Cemip',
                         'Acta2', 'Tagln', 'Myh11', 'Dgkb',
                         'Abcc9', 'Kcnj8', 'Zfp804b', 'Serpini1',
                         'Top2a', 'Pclaf', 'Diaph3', 'Mki67')

In [ ]:
expression <- as.data.frame(t(integrated_noDBL[["RNA"]]@data[markers,]))


In [ ]:
expression$Cell <- rownames(expression)
expression$Idents <- Idents(integrated_noDBL)

In [ ]:
expression <- reshape2::melt(expression, id.vars = c("Cell","Idents"), measure.vars = markers,
                       variable.name = "Feat", value.name = "Expr")

In [ ]:
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 7, repr.plot.res = 100)

# Create a grouping variable for coloring every 4 genes the same
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)

c <- ggplot(expression, aes(factor(Feat), Expr, fill = color_group)) +
        geom_violin(scale = "width", adjust = 1, trim = TRUE) +
        scale_y_continuous(expand = c(0, 0), position="right", labels = function(x)
                           c(rep(x = "", times = length(x)-2), x[length(x) - 1], "")) +
        facet_grid(rows = vars(Idents), scales = "free", switch = "y") +
        theme_cowplot(font_size = 12) +
        theme(legend.position = "none", panel.spacing = unit(0, "lines"),
          plot.title = element_text(hjust = 0.5),
              panel.background = element_rect(fill = NA, color = "black"),
              strip.background = element_blank(),
              strip.text = element_text(face = "bold"),
              strip.text.y.left = element_text(angle = 0),
              axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5)) + 
                           scale_fill_manual(values = c(my_c[1], my_c[3], my_c[2], my_c[4]))

print(c)


ggsave(file="New_Mesenchymal_noDBL_all_integrated_vln_Cell_type.svg", plot=c, width=20, height=7)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Mesenchymal_integrated_noDBL_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_Peaks')

dpt

ggsave(file="Mesenchymal_integrated_noDBL_celltype_peaks.svg", plot=dpt, width=7, height=4)

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_multiome_celltype.rds')

# Joint vizualization

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_multiome_celltype.rds')

In [ ]:
Annotation(integrated_noDBL) <- annotation

In [ ]:
integrated_noDBL@reductions

In [ ]:
integrated_noDBL <- FindMultiModalNeighbors(integrated_noDBL, reduction.list = c('harmonyRNA', 'harmonypeaks'),
                                       dims.list = list(1:11,2:14), verbose=TRUE)

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, nn.name = "weighted.nn", assay = "RNA", reduction.name = 'umap_multiome',  verbose = TRUE)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=2.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'umap_multiome')

dpt

ggsave(file="Mesenchymal_integrated_noDBL_clusters_predictedid_Multiome.svg", plot=dpt, width=7, height=5)

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_noDBL_multiome_celltype_joint.rds')

# Marker genes on different umaps assays

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint.rds')

In [ ]:
mesenchymal_noDBL_markers <- c('Vcam1', 'Rbfox1', 'Col26a1', 'Cemip',
                         'Acta2', 'Tagln', 'Myh11', 'Dgkb',
                         'Abcc9', 'Kcnj8', 'Zfp804b', 'Serpini1',
                         'Top2a', 'Pclaf', 'Diaph3', 'Mki67')

## RNA

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = mesenchymal_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_RNA_umap.svg", plot=ftp, width=20, height=20)

## Peaks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, reduction = 'harmony_Peaks', raster = TRUE, 
               pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_Peaks_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_Peaks', features = mesenchymal_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_Peaks_umap.svg", plot=ftp, width=20, height=20)

## Joint vuzualization

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, reduction = 'umap_multiome', raster = TRUE,
               pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_Joint_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'umap_multiome', features = mesenchymal_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 2.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_Joint_umap.svg", plot=ftp, width=20, height=20)

# Linking peaks to genes

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint.rds')

In [ ]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
# first compute the GC content for each peak
integrated_noDBL <- RegionStats(integrated_noDBL, genome = BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- LinkPeaks(integrated_noDBL, peak.assay = "peaks", expression.assay = "RNA")

In [ ]:
integrated_noDBL

In [ ]:
saveRDS(integrated_noDBL, 'Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

# Find DA peaks

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

## Cell types

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'
Idents(integrated_noDBL) <- "Cell_type"

In [ ]:
da_peaks <- FindAllMarkers(
  object = integrated_noDBL,
  test.use = 'LR',
  min.pct = 0.1
)

In [ ]:
da_peaks <- da_peaks[da_peaks$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks, 'DA_peaks_mesenchymal.csv')

## Condition

In [ ]:
Idents(integrated_noDBL) <- "Age"

In [ ]:
da_peaks_age <- FindAllMarkers(
  object = integrated_noDBL,
  assay = 'peaks',
  test.use = 'wilcox',
  min.pct = 0.01
)

In [ ]:
dim(da_peaks_age)

In [ ]:
da_peaks_age <- da_peaks_age[da_peaks_age$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks_age, 'DA_peaks_mesenchymal_age.csv')

In [ ]:
dim(da_peaks_age)

In [ ]:
p11_peaks <- subset(da_peaks_age, cluster == 'P11')

In [ ]:
dim(p11_peaks)

In [ ]:
p60_peaks <- subset(da_peaks_age, cluster == 'P60')

In [ ]:
dim(p60_peaks)

# Plots for figures

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
integrated_noDBL$log10_ngenes <- log10(integrated_noDBL$nFeature_RNA)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('log10_ngenes'), 
        ncol=4, pt.size=0, col = c(my_c[1], my_c[3], my_c[2], my_c[4])) + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_mesenchymal_integrated_all_vln_log10genes_Cell_type.svg", plot=vln, width=14, height=6)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('pct_mito'), 
        ncol=4, pt.size=0, col = c(my_c[1], my_c[3], my_c[2], my_c[4])) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_mesenchymal_integrated_all_vln_pct_mito_Cell_type.svg", plot=vln, width=14, height=6)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('log10_ngenes'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_mesenchymal_integrated_all_vln_log10genes_Sample.svg", plot=vln, width=14, height=6)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('pct_mito'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_mesenchymal_integrated_all_vln_pct_mito_Sample.svg", plot=vln, width=14, height=6)


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('nCount_RNA', 'log10_ngenes', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_mesenchymal_integrated_all_vln_log10genes_pct_mito_all.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1], my_c[3], my_c[2], my_c[4]), 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=7, height=4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1], my_c[3], my_c[2], my_c[4]), 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_peaks_umap.svg", plot=dmp, width=7, height=4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1], my_c[3], my_c[2], my_c[4]), 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_joint_umap.svg", plot=dmp, width=7, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Age_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Age_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 2.5)

dmp

ggsave(file="Harmony_noDBL_Age_joint_umap.svg", plot=dmp, width=6, height=4)

## Highlight DA peaks

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

### Vcam1

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Vcam1', feature = 'Vcam1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Vcam1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Acta2

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Acta2', feature = 'Acta2', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Acta2', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Kcnj8

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Kcnj8', feature = 'Kcnj8', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Kcnj8', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

## Peaks by Age

### Inmt

In [ ]:
regions <- c('chr6-55179575-55180350')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated_noDBL, region = 'Inmt', feature = 'Inmt', region.highlight = ranges.show, group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 5000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_highlight_noDBL_", 'Inmt', "_igv_tracks_mesen.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Inmt', feature = 'Inmt', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 5000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Inmt', "_igv_tracks_mesen.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Col3a1

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Col3a1', feature = 'Col3a1', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Col3a1', "_igv_tracks_mesen.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

# Xist counts to try to separate male and female cells 

In [ ]:
integrated_noDBL <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
integrated_noDBL

In [ ]:
xist_counts <- GetAssayData(integrated_noDBL, assay = "RNA", layer = "counts")[ "Xist",]

In [ ]:
integrated_noDBL$'Xist-counts' <- xist_counts

In [ ]:
integrated_noDBL$'Female' <- integrated_noDBL$`Xist-counts` > 0

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 2)

dmp

ggsave(file="Gender_inference_Harmony_noDB_RNA_umap_mesen.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 2)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_peaks_umap_mesen.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 2)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_joint_umap_mesen.svg", plot=dmp, width=6, height=4)

# Session info

In [ ]:
print(sessioninfo::session_info())